# NEXUS — Colab GPU + Ollama + Google Drive — REV3
Dias 1–5. Modelos completos do curso, persistência no Drive, fixes de SQLite e Qwen3.
Ollama: `qwen3:1.7b`, `qwen3:4b`, `qwen3:8b`, `nomic-embed-text`.
HF Dia 5: E5-small, mMARCO reranker, mDeBERTa zero-shot, XLM-R QA, Qwen3-0.6B e rembg/u2net.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os,sys,shutil,subprocess,time,json,re,uuid,importlib,platform,requests
D=Path("/content/drive/MyDrive/minicurso-mult-agents-colab");DR=D/"repo";DS=D/"state";OM=D/"ollama/models";HF=D/"huggingface";RB=D/"rembg"
R=Path("/content/minicurso-mult-agents");N=R/"codigo/nexus"
for p in (D,DS,OM,HF,RB):p.mkdir(parents=True,exist_ok=True)
def run(c,check=True,capture=False,env=None,cwd=None):return subprocess.run(c,shell=isinstance(c,str),check=check,text=True,capture_output=capture,env=env,cwd=cwd)
g=run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],False,True)
if g.returncode:raise RuntimeError("Selecione GPU no runtime do Colab")
print(g.stdout.strip())

In [ ]:
URL="https://github.com/overcyber/minicurso-mult-agents.git"
if not (DR/".git").exists():run(["git","clone",URL,str(DR)])
elif not run(["git","-C",str(DR),"status","--porcelain"],capture=True).stdout.strip():run(["git","-C",str(DR),"pull","--ff-only","origin","main"])
if R.exists():shutil.rmtree(R)
shutil.copytree(DR,R,ignore=shutil.ignore_patterns(".git","__pycache__","*.pyc",".chroma",".chroma_hf",".chroma_hf_gpu","*.db","saida","tracos"))
os.chdir(N)
cons=Path("/content/nexus-constraints.txt");cons.write_text("langchain==1.4.0\nlanggraph==1.2.11\nlanggraph-checkpoint-sqlite==3.1.1\nlangchain-ollama==1.1.0\nlangchain-chroma==1.1.0\nchromadb==1.5.9\n")
run([sys.executable,"-m","pip","install","-q","-r",str(N/"requirements.txt"),"-r",str(N/"dia5/space_demo/requirements.txt"),"-c",str(cons),"accelerate>=1.0"])
os.environ.update(HF_HOME=str(HF),HF_HUB_CACHE=str(HF/"hub"),TRANSFORMERS_CACHE=str(HF/"transformers"),SENTENCE_TRANSFORMERS_HOME=str(HF/"sentence-transformers"),REMBG_HOME=str(RB),OLLAMA_MODELS=str(OM),OLLAMA_HOST="127.0.0.1:11434")

In [ ]:
OLLAMA={"small":"qwen3:1.7b","base":"qwen3:4b","medium":"qwen3:4b","large":"qwen3:8b","embedding":"nomic-embed-text"}
HFM={"embedding":"intfloat/multilingual-e5-small","reranker":"cross-encoder/mmarco-mMiniLMv2-L12-H384-v1","sentiment":"nlptown/bert-base-multilingual-uncased-sentiment","zero_shot":"MoritzLaurer/mDeBERTa-v3-base-mnli-xnli","qa":"deepset/xlm-roberta-base-squad2","chat":"Qwen/Qwen3-0.6B"}
VLLM_OPTIONAL="Qwen/Qwen3-8B";REMBG_MODEL="u2net"
os.environ.update(MODELO=OLLAMA["base"],MODELO_PEQUENO=OLLAMA["small"],MODELO_MEDIO=OLLAMA["medium"],MODELO_GRANDE=OLLAMA["large"])
print("Ollama",OLLAMA);print("HF",HFM)

In [ ]:
def install_ollama():
    if shutil.which("ollama"):return
    run("apt-get update -qq",False);run("apt-get install -y -qq curl tar zstd ca-certificates")
    x=run("curl -fsSL https://ollama.com/install.sh | sh",False,True)
    if x.returncode==0 and shutil.which("ollama"):return
    a={"x86_64":"amd64","amd64":"amd64","aarch64":"arm64","arm64":"arm64"}.get(platform.machine().lower())
    if not a:raise RuntimeError("arquitetura sem fallback")
    x=run(f"curl -fsSL https://ollama.com/download/ollama-linux-{a}.tar.zst | tar --zstd -x -C /usr",False,True)
    if x.returncode or not shutil.which("ollama"):raise RuntimeError(x.stdout+x.stderr)
install_ollama()
def alive():
    try:return requests.get("http://127.0.0.1:11434/api/tags",timeout=2).ok
    except:return False
if not alive():
    log=open(N/"ollama.log","ab",buffering=0);proc=subprocess.Popen(["ollama","serve"],env=os.environ.copy(),stdout=log,stderr=subprocess.STDOUT)
    for _ in range(60):
        if alive():break
        time.sleep(1)
if not alive():raise RuntimeError("Ollama não iniciou")

In [ ]:
def canon(x):
    x=(x or "").strip().lower();return x[:-7] if x.endswith(":latest") else x
def tags():return {m["name"] for m in requests.get("http://127.0.0.1:11434/api/tags",timeout=10).json().get("models",[])}
for m in dict.fromkeys(OLLAMA.values()):
    if canon(m) not in {canon(x) for x in tags()}:run(["ollama","pull",m],env=os.environ.copy())
assert {canon(x) for x in OLLAMA.values()} <= {canon(x) for x in tags()}
print(sorted(tags()))
print(run(["ollama","run",OLLAMA["base"],"Responda apenas OK."],capture=True,env=os.environ.copy()).stdout)
print(run(["ollama","ps"],False,True,env=os.environ.copy()).stdout)

In [ ]:
from huggingface_hub import snapshot_download
PREFETCH_HF=True
if PREFETCH_HF:
    for repo in dict.fromkeys(HFM.values()):
        print("[HF]",repo);snapshot_download(repo_id=repo,cache_dir=os.environ["HF_HUB_CACHE"],ignore_patterns=["onnx/*","openvino/*","*.tflite","*.h5","tf_model.*","flax_model.*"])
from rembg import new_session
_s=new_session(REMBG_MODEL);del _s

In [ ]:
STATE_DIRS=[".chroma",".chroma_hf",".chroma_hf_gpu","saida","tracos"]
def cp(a,b):
    if not a.exists():return
    if a.is_dir():
        if b.exists():shutil.rmtree(b)
        shutil.copytree(a,b)
    else:b.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(a,b)
def restore():
    for x in STATE_DIRS:cp(DS/x,N/x)
    for pat in ("*.db","*.db-wal","*.db-shm","*.csv","*.json","*.jsonl","ollama.log"):
        for a in DS.glob(pat):cp(a,N/a.name)
def persist():
    for x in STATE_DIRS:cp(N/x,DS/x)
    for pat in ("*.db","*.db-wal","*.db-shm","*.csv","*.json","*.jsonl","ollama.log"):
        for a in N.glob(pat):cp(a,DS/a.name)
restore()
MODS={"agente","clientes","ferramentas","indexar","nexus","hooks","steering","ferramentas_web","equipe","prompts","handoff","memoria_semantica","email_assistente","app_gradio","avaliacao","modelos","roteador","pipelines","embeddings_hf","cli"}
def dia(n):
    for m in list(sys.modules):
        if m in MODS:sys.modules.pop(m,None)
    ps=[str(N/f"dia{i}") for i in range(1,6)];sys.path[:]=[p for p in sys.path if p not in ps];sys.path.insert(0,str(N/n));importlib.invalidate_caches();os.chdir(N)
run([sys.executable,"-m","compileall","-q",str(N)])
t=run([sys.executable,"-m","pytest","-q",str(N/"testes")],False,True,cwd=str(N));print(t.stdout)
if t.returncode:raise RuntimeError(t.stderr)

# Dia 1

In [ ]:
dia("dia1");import ferramentas as f1;print(f1.calcular("950 * 24"));print(f1.ler_arquivo("../../etc/passwd"))
import agente as a1;print(a1.rodar("Qual foi o faturamento de 2024? Leia o documento necessário e cite o nome dele.",backend="ollama",verbose=True))

# Dia 2

In [ ]:
dia("dia2");import indexar as i2
b=i2.construir(recriar=not(N/".chroma").exists())
for d in b.similarity_search("faturamento de 2024",k=2):print(Path(d.metadata.get("source","?")).name,d.page_content[:400])
import agente as a2;print(a2.rodar("Qual foi o faturamento de 2024? Cite a fonte."))

# Dia 3

In [ ]:
dia("dia3");import hooks,steering;print(hooks.avaliar_politicas("ler_arquivo",{"caminho":"../../etc/passwd"},{}));print(steering.detectar_estagnacao({"passos":7,"achados":[]}))
from langchain_core.messages import HumanMessage
import nexus as n3,sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
conn=sqlite3.connect(str(N/"nexus_colab.db"),check_same_thread=False)
try:
    g=n3.construir_grafo().compile(checkpointer=SqliteSaver(conn),interrupt_before=[])
    r=g.invoke({"messages":[HumanMessage("Qual foi o faturamento de 2024? Cite a fonte.")],"passos":0},{"configurable":{"thread_id":"colab-d3"},"recursion_limit":30});print(r["messages"][-1].content)
finally:conn.close()
persist()

# Dia 4

In [ ]:
dia("dia4");from langchain_core.messages import HumanMessage;import equipe as e4,sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
conn=sqlite3.connect(str(N/"equipe_colab.db"),check_same_thread=False)
try:
    g=e4.construir_equipe().compile(checkpointer=SqliteSaver(conn));q="Compare os fornecedores e recomende um, citando fontes."
    e={"messages":[HumanMessage(q)],"pergunta":q,"proximo":"","instrucao":q,"achados":[],"rascunho":"","veredito":"","rodadas":0}
    r=g.invoke(e,{"configurable":{"thread_id":"colab-d4"},"recursion_limit":40});print(r.get("veredito"));print(r.get("rascunho") or r["messages"][-1].content)
finally:conn.close()
persist()

In [ ]:
dia("dia2");import indexar as ix;be=ix.construir(False)
dia("dia4");import email_assistente as ea,sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command
conn=sqlite3.connect(str(N/"email_colab.db"),check_same_thread=False)
try:
    ge=ea.construir_grafo(retriever=be);ge.checkpointer=SqliteSaver(conn);emails=json.loads((N/"dados/emails.json").read_text());cfg={"configurable":{"thread_id":"email-0"}};st=ge.invoke(emails[0],cfg);snap=ge.get_state(cfg)
    if snap.tasks and snap.tasks[0].interrupts:st=ge.invoke(Command(resume={"acao":"descartar"}),cfg)
    print(st)
finally:conn.close()
persist()

# Dia 5

In [ ]:
import torch;dev="cuda" if torch.cuda.is_available() else "cpu";pdev=0 if torch.cuda.is_available() else -1
if dev!="cuda":raise RuntimeError("CUDA indisponível")
dia("dia5");import modelos as m5
exp={"supervisor":OLLAMA["small"],"pesquisador":OLLAMA["medium"],"analista":OLLAMA["small"],"redator":OLLAMA["large"],"critico":OLLAMA["medium"]}
for p,x in exp.items():print(p,m5.para(p).model);assert m5.para(p).model==x
from langchain_huggingface import HuggingFaceEmbeddings
emb=HuggingFaceEmbeddings(model_name=HFM["embedding"],model_kwargs={"device":dev},encode_kwargs={"normalize_embeddings":True});print(len(emb.embed_query("faturamento 2024")))

In [ ]:
from transformers import pipeline,AutoModelForCausalLM,AutoTokenizer
sent=pipeline("sentiment-analysis",model=HFM["sentiment"],device=pdev);print(sent(["Excelente equipamento","Atendimento péssimo"],truncation=True))
dia("dia5");import pipelines as p5
z=pipeline("zero-shot-classification",model=HFM["zero_shot"],device=pdev);print(z("Preciso comprar um torno industrial",candidate_labels=p5.CATEGORIAS))
qa=pipeline("question-answering",model=HFM["qa"],device=pdev);ctx=(N/"dados/faq/politicas.md").read_text();print(qa(question="Qual é a política descrita?",context=ctx))
tok=AutoTokenizer.from_pretrained(HFM["chat"]);model=AutoModelForCausalLM.from_pretrained(HFM["chat"],torch_dtype="auto",device_map="auto")
msgs=[{"role":"system","content":"Responda em português e objetivamente."},{"role":"user","content":"Explique em duas frases o papel de um supervisor multiagente."}]
inp=tok.apply_chat_template(msgs,tokenize=True,add_generation_prompt=True,enable_thinking=False,return_tensors="pt",return_dict=True);inp={k:v.to(model.device) for k,v in inp.items()}
with torch.inference_mode():out=model.generate(**inp,max_new_tokens=180,do_sample=False,pad_token_id=tok.eos_token_id)
txt=tok.decode(out[0][inp["input_ids"].shape[-1]:],skip_special_tokens=True).strip();assert "<think>" not in txt;print(txt)

In [ ]:
# Avaliação completa opcional, com SQLite vivo durante o lote.
RODAR_10=False
dia("dia4");import equipe as ev,sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_core.messages import HumanMessage
conn=None
def responder(q):
    global conn
    if conn is None:conn=sqlite3.connect(str(N/"equipe_avaliacao_colab.db"),check_same_thread=False)
    g=ev.construir_equipe().compile(checkpointer=SqliteSaver(conn))
    e={"messages":[HumanMessage(q)],"pergunta":q,"proximo":"","instrucao":q,"achados":[],"rascunho":"","veredito":"","rodadas":0}
    r=g.invoke(e,{"configurable":{"thread_id":f"eval-{uuid.uuid4()}"},"recursion_limit":40});txt=r.get("rascunho") or r["messages"][-1].content or "";fontes=sorted(set(re.findall(r"\[fonte:\s*([^\]]+)\]",txt,re.I)));return txt,fontes,0
dia("dia5");import avaliacao as av;casos=av.carregar_casos(N/"avaliacao/casos.jsonl");print("casos",len(casos))
try:
    if RODAR_10:
        linhas=av.rodar(responder,casos);print(av.tabela(linhas))
finally:
    if conn is not None:conn.close()
persist()

In [ ]:
persist();print("Drive:",D);print(run(["ollama","ps"],False,True,env=os.environ.copy()).stdout)